# TRUST run1 — Google Colab, batch size 32

Notebook này chỉ giữ **một cấu hình run1 chính**: DINOv3 + GroundingDINO/Faster R-CNN,
full DeBERTa fine-tuning, NCOD/LUM/HUM, verifier, hard-negative mining, EMA và cosine warmup.
LoRA, multi-profile tuning và causal knowledge JSON/memory gate đã được loại khỏi pipeline.

## Before you run

1. Chọn **Runtime → Change runtime type → GPU**.
2. Mở **Colab → Secrets** và thêm `KAGGLE_API_TOKEN`, hoặc upload `~/.kaggle/kaggle.json` trước cell xác thực.
   Bật quyền notebook truy cập secret này. Không ghi token trực tiếp vào notebook.
3. Nếu dùng W&B online, thêm secret `WANDB_API_KEY` và giữ
   `WANDB_MODE_SETTING = 'online'` ở CELL 2.
4. Nếu không muốn upload W&B, đổi thành `WANDB_MODE_SETTING = 'offline'`. Offline run vẫn
   được lưu local và có thể sync sau bằng `wandb sync`.
5. Chạy notebook tuần tự từ CELL 1. Chỉ tiếp tục sau khi CELL 2 in
   `Dependency/auth preflight OK` và CELL 6 in object shape `(16, 12, 2820)`.

## Cấu hình run1 cố định

```text
epochs=10 | batch=32 | accumulation=1 | effective_batch=32
main_lr=1e-5 | text_lr=5e-6 | weight_decay=1e-4
lambda_verifier=0.25 | knowledge=disabled
EMA=0.999 | NCOD_U_lr=0.1 | HUM_alpha=1.0
frame input=16 | selected frames=5 | objects/frame=12 | object dim=2820
num_workers=0 | pin_memory=False
```

Kết quả local nằm trong `/content/working/models`. Weight chỉ được upload một lần ở cuối run:
một artifact `last` và một artifact `best`.


In [ ]:
# CELL 1: Git Clone & Setup (Colab, original n2 branch)
import os
from pathlib import Path

REPO_URL = 'https://github.com/DanielQH07/tranSTR_Casual.git'
REPO_NAME = 'tranSTR_Casual'
BRANCH = 'master'
WORKING_DIR = Path('/content')
REPO_DIR = WORKING_DIR / REPO_NAME

def has_repo_files(p):
    p = Path(p)
    return (p / 'DataLoader.py').exists() and (p / 'networks' / 'model.py').exists()

if not has_repo_files(Path.cwd()):
    if not REPO_DIR.exists():
        rc = os.system(f'git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}')
        if rc != 0:
            raise RuntimeError('git clone failed')
    target_dir = REPO_DIR / 'causalvid'
    os.chdir(target_dir if target_dir.exists() else REPO_DIR)
if not has_repo_files(Path.cwd()):
    raise FileNotFoundError(f'Repo files not found in {Path.cwd()}')
print(f'Working directory: {Path.cwd()}')


In [ ]:
%cd /content/tranSTR_Casual

In [ ]:
# CELL 2: Dependencies, Kaggle auth, and W&B mode
print('=== CELL 2: Dependencies & Authentication ===')
import importlib
import os
import subprocess
import sys

# Choose exactly one mode before running this cell: 'online' or 'offline'.
WANDB_MODE_SETTING = 'online'
if WANDB_MODE_SETTING not in {'online', 'offline'}:
    raise ValueError("WANDB_MODE_SETTING must be 'online' or 'offline'")

REQUIRED_PIP_PACKAGES = [
    'wandb', 'kagglehub', 'transformers', 'sentencepiece', 'einops', 'tqdm'
]
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q', '-U', *REQUIRED_PIP_PACKAGES
])

try:
    from google.colab import userdata
except Exception:
    userdata = None

def _read_secret(name):
    value = os.environ.get(name, '').strip()
    if value:
        return value
    if userdata is not None:
        try:
            return (userdata.get(name) or '').strip()
        except Exception:
            return ''
    return ''

# KaggleHub accepts either KAGGLE_API_TOKEN or the legacy ~/.kaggle/kaggle.json file.
kaggle_token = _read_secret('KAGGLE_API_TOKEN')
kaggle_json_path = os.path.expanduser('~/.kaggle/kaggle.json')
if kaggle_token:
    os.environ['KAGGLE_API_TOKEN'] = kaggle_token
    print('Kaggle auth: KAGGLE_API_TOKEN')
elif os.path.isfile(kaggle_json_path):
    print(f'Kaggle auth: {kaggle_json_path}')
else:
    raise RuntimeError(
        'Missing Kaggle credentials. Add Colab Secret KAGGLE_API_TOKEN or upload '
        '~/.kaggle/kaggle.json with chmod 600 before running this cell.'
    )

wandb_key = _read_secret('WANDB_API_KEY')
if WANDB_MODE_SETTING == 'online':
    if not wandb_key:
        raise RuntimeError(
            "W&B online selected but WANDB_API_KEY is missing. Add the Colab secret "
            "or set WANDB_MODE_SETTING='offline'."
        )
    os.environ['WANDB_API_KEY'] = wandb_key
    os.environ.pop('WANDB_MODE', None)
else:
    os.environ['WANDB_MODE'] = 'offline'

REQUIRED_IMPORT_MODULES = [
    'torch', 'numpy', 'pandas', 'tqdm', 'IPython',
    'einops', 'transformers', 'wandb', 'kagglehub'
]
missing = []
for module_name in REQUIRED_IMPORT_MODULES:
    try:
        importlib.import_module(module_name)
    except Exception as exc:
        missing.append(f'{module_name}: {exc}')
if missing:
    raise ImportError('Dependency preflight failed:\n' + '\n'.join(missing))

import wandb
if WANDB_MODE_SETTING == 'online':
    wandb.login(key=wandb_key, verify=True)
    print('W&B mode: online')
else:
    print('W&B mode: offline')

WANDB_PROJECT = 'trust-causal-videoqa'
WANDB_ENTITY = None
print('Dependency/auth preflight OK')


In [ ]:
# CELL 3: Resolve and merge data paths with KaggleHub (Google Colab)
print('=== CELL 3: Data Paths (KaggleHub) ===')
import os, shutil
from pathlib import Path
import kagglehub

def _download(slug, env_name=None):
    override = os.environ.get(env_name, '').strip() if env_name else ''
    return Path(override) if override and Path(override).exists() else Path(kagglehub.dataset_download(slug))
def _find_dir_with_ext(root, ext):
    counts = {}
    for p in Path(root).rglob('*' + ext):
        counts[str(p.parent)] = counts.get(str(p.parent), 0) + 1
    return Path(max(counts, key=counts.get)) if counts else None
def _find_named(root, name):
    root = Path(root)
    if root.name.lower() == name.lower():
        return root
    return next((p for p in root.rglob('*') if p.is_dir() and p.name.lower() == name.lower()), None)

dinov3_root = _download('danielq07/dinov3-feat', 'DINOV3_DATASET_PATH')
gdino_root = _download('danielq07/causal-vidqa-gdinofasterrcnn-features-merged', 'GDINO_DATASET_PATH')
anno_root = _download('lusnaw/text-annotation', 'ANNO_DATASET_PATH')
split_root = _download('danielq07/casual-vid-data-split', 'SPLIT_DATASET_PATH')

all_dinov3_pt = list(Path(dinov3_root).rglob('*.pt'))
if not all_dinov3_pt:
    raise FileNotFoundError(f'No DINOv3 .pt files found under {dinov3_root}')

GDINO_MERGED_PATH = _find_dir_with_ext(gdino_root, '.pkl') or gdino_root
ANNOTATION_QA_PATH = _find_named(anno_root, 'QA') or anno_root
SPLIT_TXT_PATH = _find_named(split_root, 'split') or split_root

# DataLoader expects one flat directory. KaggleHub/Colab cache keeps DINOv3
# under train/valid/test, so link all three splits into a single view.
CLIP_MERGED_PATH = Path('/content/dinov3_T16_dim1024_merge')
CLIP_MERGED_PATH.mkdir(parents=True, exist_ok=True)
for src in all_dinov3_pt:
    dst = CLIP_MERGED_PATH / src.name
    if dst.exists():
        continue
    try:
        dst.symlink_to(src)
    except Exception:
        shutil.copy2(src, dst)

merged_count = len(list(CLIP_MERGED_PATH.glob('*.pt')))
source_unique_count = len({src.name for src in all_dinov3_pt})
if merged_count != source_unique_count:
    raise RuntimeError(
        f'DINOv3 merge incomplete: merged={merged_count}, source_unique={source_unique_count}'
    )
print(f'DINOv3 merged all splits: {merged_count} files')
print('Resolved paths:', CLIP_MERGED_PATH, GDINO_MERGED_PATH, ANNOTATION_QA_PATH, SPLIT_TXT_PATH)
for name, path in [('DINOv3', CLIP_MERGED_PATH), ('GDINO+FRCNN', GDINO_MERGED_PATH), ('QA', ANNOTATION_QA_PATH), ('split', SPLIT_TXT_PATH)]:
    if not Path(path).exists(): raise FileNotFoundError(f'{name} path not found: {path}')


In [ ]:
# CELL 4: Core imports + run1 NCOD/HUM/verifier functions
print('=== CELL 4: Imports + Functions (run1 NCOD + HUM + Verifier) ===')

import copy
import json
import math
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from IPython.display import display

from utils.util import set_seed, set_gpu_devices
from DataLoader import VideoQADataset
from networks.model import VideoQAmodel

def _unpack_batch(batch):
    if len(batch) == 7:
        ff, of, q, a, ans_id, qns_key, q_family_id = batch
    elif len(batch) == 6:
        ff, of, q, a, ans_id, qns_key = batch
        q_family_id = None
    else:
        raise ValueError(f'Unexpected batch format with {len(batch)} elements')
    return ff, of, q, a, ans_id, qns_key, q_family_id

def _compute_sample_indices(qns_keys, key_to_idx, device):
    idxs = [key_to_idx.get(str(k), -1) for k in qns_keys]
    if any(i < 0 for i in idxs):
        missing = [str(qns_keys[i]) for i, v in enumerate(idxs) if v < 0][:5]
        raise KeyError(f'Missing qns_key in key_to_idx mapping: {missing}')
    return torch.tensor(idxs, dtype=torch.long, device=device)

_QTYPE_SUFFIXES = [
    'counterfactual_reason',
    'predictive_reason',
    'counterfactual',
    'predictive',
    'explanatory',
    'descriptive',
]

def _split_qns_key(qns_key):
    key = str(qns_key)
    for qtype in _QTYPE_SUFFIXES:
        suffix = f'_{qtype}'
        if key.endswith(suffix):
            return key[:-len(suffix)], qtype
    return key, 'unknown'

def _compute_acc_all_metrics(type_results):
    mapping = [
        ('Description', 'descriptive'),
        ('Explanation', 'explanatory'),
        ('Predictive-Answer', 'predictive'),
        ('Predictive-Reason', 'predictive_reason'),
        ('Counterfactual-Answer', 'counterfactual'),
        ('Counterfactual-Reason', 'counterfactual_reason'),
    ]
    metrics = {}
    for name, qtype in mapping:
        rows = type_results.get(qtype, [])
        metrics[name] = (sum(1 for r in rows if r['correct']) / len(rows) * 100) if rows else 0.0

    def _hard_metric(type_ans, type_reason):
        ans_by_vid = {r['video_id']: r['correct'] for r in type_results.get(type_ans, [])}
        reason_by_vid = {r['video_id']: r['correct'] for r in type_results.get(type_reason, [])}
        common_vids = set(ans_by_vid) & set(reason_by_vid)
        if not common_vids:
            return 0.0
        both_correct = sum(1 for vid in common_vids if ans_by_vid[vid] and reason_by_vid[vid])
        return both_correct / len(common_vids) * 100

    metrics['PAR'] = _hard_metric('predictive', 'predictive_reason')
    metrics['CAR'] = _hard_metric('counterfactual', 'counterfactual_reason')
    metrics['Acc_ALL'] = (metrics['Description'] + metrics['Explanation'] + metrics['PAR'] + metrics['CAR']) / 4.0
    return metrics

def _hard_negative_weights(cand_feat, target, enabled=False, max_weight=1.5):
    if (not enabled) or cand_feat is None or max_weight <= 1.0:
        return torch.ones_like(target, dtype=torch.float32)
    with torch.no_grad():
        cand_norm = F.normalize(cand_feat.detach(), dim=-1)
        gold = cand_norm.gather(1, target.view(-1, 1, 1).expand(-1, 1, cand_norm.size(-1))).squeeze(1)
        sims = torch.bmm(cand_norm, gold.unsqueeze(-1)).squeeze(-1)
        sims.scatter_(1, target.view(-1, 1), -1.0)
        hardness = sims.max(dim=1).values.clamp(min=0.0, max=1.0)
        return 1.0 + (float(max_weight) - 1.0) * hardness

def _update_ema_model(ema_model, model, decay):
    if ema_model is None:
        return
    with torch.no_grad():
        for ema_param, param in zip(ema_model.parameters(), model.parameters()):
            ema_param.data.mul_(decay).add_(param.data, alpha=1.0 - decay)
        for ema_buffer, buffer in zip(ema_model.buffers(), model.buffers()):
            ema_buffer.copy_(buffer)

def train_epoch_integrated(
    model, optimizer_model, optimizer_u, U, loader, bce, device, epoch, key_to_idx,
    accumulation_steps=1, hum_alpha=1.0, lambda_verifier=0.25,
    use_hard_neg_mining=False, hard_neg_weight_max=1.5,
    ema_model=None, ema_decay=0.999,
    scheduler=None, scheduler_step_per_batch=False
):
    model.train()
    total_loss, total_l1, total_l2 = 0.0, 0.0, 0.0
    total_verifier = 0.0
    correct, total = 0, 0
    optimizer_model.zero_grad()
    optimizer_u.zero_grad()

    pbar = tqdm(loader, desc=f'Epoch {epoch}', leave=False)
    for batch_idx, batch in enumerate(pbar):
        ff, of, q, a, ans_id, qns_keys, q_family_id = _unpack_batch(batch)
        ff, of, tgt = ff.to(device), of.to(device), ans_id.to(device)

        if q_family_id is None:
            q_family_id = torch.zeros_like(tgt)
        else:
            q_family_id = q_family_id.to(device)

        sample_indices = _compute_sample_indices(qns_keys, key_to_idx, device)
        out = model(ff, of, q, a, return_aux=True, q_family_id=None)
        logits = out['logits']
        verifier_logits = out.get('verifier_logits', logits)
        cand_feat = out.get('cand_feat', None)

        probs = torch.softmax(logits, dim=1)
        y_onehot = F.one_hot(tgt, num_classes=logits.size(-1)).float()
        u_batch = U[sample_indices].unsqueeze(1)

        ce_per_sample = -torch.sum(y_onehot * torch.log(torch.clamp(probs, min=1e-12)), dim=1)
        shifted_probs = torch.clamp(probs + (u_batch.detach() * y_onehot), min=1e-12, max=1.0)
        lum_loss = -torch.sum(y_onehot * torch.log(shifted_probs), dim=1)
        hum_loss = (1.0 + hum_alpha * u_batch.detach().squeeze(1)) * ce_per_sample

        hard_neg_w = _hard_negative_weights(
            cand_feat,
            tgt,
            enabled=use_hard_neg_mining,
            max_weight=hard_neg_weight_max,
        )
        hard_mask = (q_family_id >= 2)
        l1_per_sample = torch.where(hard_mask, hum_loss, lum_loss)
        l1 = (l1_per_sample * hard_neg_w).mean()

        verifier_loss = bce(verifier_logits, y_onehot)
        model_loss = l1 + lambda_verifier * verifier_loss
        (model_loss / accumulation_steps).backward()

        shifted_det = probs.detach() + (u_batch * y_onehot)
        l2 = F.mse_loss(shifted_det, y_onehot)
        (l2 / accumulation_steps).backward()

        if (batch_idx + 1) % accumulation_steps == 0 or (batch_idx + 1) == len(loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer_model.step()
            _update_ema_model(ema_model, model, ema_decay)
            if scheduler is not None and scheduler_step_per_batch:
                scheduler.step()
            optimizer_model.zero_grad()
            optimizer_u.step()
            optimizer_u.zero_grad()
            with torch.no_grad():
                U.clamp_(0.0, 0.99)

        total_l1 += l1.item()
        total_l2 += l2.item()
        total_verifier += verifier_loss.item()
        total_loss += (model_loss + l2).item()
        correct += (logits.argmax(-1) == tgt).sum().item()
        total += tgt.size(0)

        pbar.set_postfix({
            'loss': total_loss / (batch_idx + 1),
            'l1': total_l1 / (batch_idx + 1),
            'l2': total_l2 / (batch_idx + 1),
            'ver': total_verifier / (batch_idx + 1),
            'acc': correct / max(total, 1) * 100
        })

    n = len(loader)
    return (
        total_loss / n,
        total_l1 / n,
        total_l2 / n,
        total_verifier / n,
        correct / max(total, 1) * 100
    )

def eval_epoch(model, loader, device):
    model.eval()
    correct, total = 0, 0
    type_results = {}
    with torch.no_grad():
        for batch in loader:
            ff, of, q, a, ans_id, qns_keys, q_family_id = _unpack_batch(batch)
            ff, of, tgt = ff.to(device), of.to(device), ans_id.to(device)
            q_family_id = q_family_id.to(device) if q_family_id is not None else None
            out = model(ff, of, q, a, return_aux=True, q_family_id=None)
            logits = out['logits']
            preds = logits.argmax(-1)
            correct += (preds == tgt).sum().item()
            total += tgt.size(0)

            for key, pred, target in zip(qns_keys, preds.detach().cpu().tolist(), tgt.detach().cpu().tolist()):
                video_id, qtype = _split_qns_key(key)
                type_results.setdefault(qtype, []).append({
                    'video_id': video_id,
                    'correct': bool(int(pred) == int(target)),
                })

    metrics = _compute_acc_all_metrics(type_results)
    metrics['Plain_Acc'] = correct / max(total, 1) * 100
    return metrics

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print('Imports and functions defined for integrated training.')

In [ ]:
# CELL 5: Fixed run1 configuration (Google Colab)
print('=== CELL 5: Paths & Fixed Run1 Config ===')

CLIP_FEATURE_PATH = str(CLIP_MERGED_PATH)
GDINO_FEATURE_PATH = str(GDINO_MERGED_PATH)
ANNOTATION_PATH = str(ANNOTATION_QA_PATH)
SPLIT_DIR = str(SPLIT_TXT_PATH)

BASE = '/content/working' if os.path.exists('/content') else os.getcwd()
MODEL_DIR = os.path.join(BASE, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

def verify_path(name, path):
    if not os.path.exists(path):
        raise FileNotFoundError(f'{name} not found: {path}')
    print(f'OK {name}: {path}')

verify_path('DINOv3 features', CLIP_FEATURE_PATH)
verify_path('GDINO+FRCNN features', GDINO_FEATURE_PATH)
verify_path('QA annotations', ANNOTATION_PATH)
verify_path('data splits', SPLIT_DIR)

import glob as _glob
n_pt = len(_glob.glob(os.path.join(CLIP_FEATURE_PATH, '*.pt')))
n_pkl = len(_glob.glob(os.path.join(GDINO_FEATURE_PATH, '*.pkl')))
if n_pt == 0 or n_pkl == 0:
    raise RuntimeError(f'Feature directories are empty: DINOv3={n_pt}, objects={n_pkl}')
print(f'DINOv3 .pt: {n_pt} | GDINO+FRCNN .pkl: {n_pkl}')

RUN_TRAINING = True
RUN_TAG = 'run1_colab_bs32_ncod_hum_verifier'

MODEL_FILENAME = f'best_model_{RUN_TAG}.ckpt'
LATEST_CKPT_FILENAME = f'last_checkpoint_{RUN_TAG}.ckpt'
TRAIN_HISTORY_FILENAME = f'train_history_{RUN_TAG}.csv'
PREDICTIONS_CSV_FILENAME = f'test_predictions_{RUN_TAG}.csv'
METRICS_JSON_FILENAME = f'final_metrics_{RUN_TAG}.json'
BEST_ARTIFACT_NAME = f'best-model-{RUN_TAG}'
LAST_ARTIFACT_NAME = f'last-checkpoint-{RUN_TAG}'
FINAL_ARTIFACT_NAME = f'final-results-{RUN_TAG}'

class Config:
    # Data
    video_feature_root = CLIP_FEATURE_PATH
    grounding_dino_path = GDINO_FEATURE_PATH
    sample_list_path = ANNOTATION_PATH
    split_dir_txt = SPLIT_DIR
    topK_frame = 16
    objs = 12
    select_frames = 5
    frame_feat_dim = 1024
    obj_feat_dim = 2820
    use_grounding_dino = True

    # Object/frame path actually used by run1
    obj_use_bbox_pos_embed = True
    obj_bbox_dim = 4
    obj_hard_gather_from_frame = True
    hard_eval = False

    # Transformer and text encoder
    d_model = 768
    nheads = 8
    num_encoder_layers = 2
    normalize_before = True
    activation = 'gelu'
    dropout = 0.3
    encoder_dropout = 0.3
    text_encoder_type = 'microsoft/deberta-base'
    freeze_text_encoder = False
    text_encoder_lr = 5e-6

    # Fixed run1 training
    bs = 32
    accumulation_steps = 1
    lr = 1e-5
    epoch = 10
    gpu = 0
    decay = 1e-4
    n_query = 5
    return_family_id = True
    lambda_verifier = 0.25
    aux_warmup_epochs = 2

    # Scheduler, early stopping, hard negatives, EMA, NCOD/HUM
    lr_schedule = 'cosine_warmup'
    warmup_epochs = 1
    min_lr = 1e-7
    early_stop_patience = 4
    early_stop_min_delta = 0.05
    early_stop_start_epoch = 5
    use_hard_neg_mining = True
    hard_neg_weight_max = 1.5
    use_ema = True
    ema_decay = 0.999
    ncod_u_lr = 0.1
    hum_alpha = 1.0

args = Config()
set_gpu_devices(args.gpu)
set_seed(999)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type != 'cuda':
    raise RuntimeError('GPU runtime is required. Select a GPU in Colab Runtime settings.')

print(f'Device: {device} | GPU: {torch.cuda.get_device_name(0)}')
print(f'Run tag: {RUN_TAG}')
print(f'Frames: input={args.topK_frame}, selected={args.select_frames}')
print(f'Objects: count={args.objs}, dim={args.obj_feat_dim}, bbox_pos={args.obj_use_bbox_pos_embed}')
print(f'Batch: {args.bs} x accumulation {args.accumulation_steps} = {args.bs * args.accumulation_steps}')
print(f'LR: main={args.lr}, text={args.text_encoder_lr}, decay={args.decay}')
print(f'Run1: verifier={args.lambda_verifier}, knowledge=disabled, EMA={args.ema_decay}')


In [ ]:
# CELL 6: Create Datasets
print('=== CELL 6: Datasets ===')

train_ds = VideoQADataset(
    split='train', n_query=args.n_query, obj_num=args.objs,
    sample_list_path=args.sample_list_path,
    video_feature_path=args.video_feature_root,
    grounding_dino_path=args.grounding_dino_path,
    split_dir=args.split_dir_txt, topK_frame=args.topK_frame,
    max_samples=None, verbose=True, return_family_id=args.return_family_id
)
val_ds = VideoQADataset(
    split='val', n_query=args.n_query, obj_num=args.objs,
    sample_list_path=args.sample_list_path,
    video_feature_path=args.video_feature_root,
    grounding_dino_path=args.grounding_dino_path,
    split_dir=args.split_dir_txt, topK_frame=args.topK_frame,
    max_samples=None, verbose=True, return_family_id=args.return_family_id
)
test_ds = VideoQADataset(
    split='test', n_query=args.n_query, obj_num=args.objs,
    sample_list_path=args.sample_list_path,
    video_feature_path=args.video_feature_root,
    grounding_dino_path=args.grounding_dino_path,
    split_dir=args.split_dir_txt, topK_frame=args.topK_frame,
    max_samples=None, verbose=True, return_family_id=args.return_family_id
)

# Final Colab guard: every object sample becomes [T=16, O=12, D=2820].
# The original branch already builds 2048 ROI + 768 class + 4 bbox features;
# this additionally supports legacy pickle files whose axes are short/long.
from torch.utils.data._utils.collate import default_collate

def _fit_object_feature_sample(sample):
    items = list(sample)
    obj = torch.as_tensor(items[1]).float()
    expected = (args.topK_frame, args.objs, args.obj_feat_dim)
    if obj.ndim != 3:
        raise RuntimeError(f'Expected object feature [T,O,D], got {tuple(obj.shape)}')
    fixed = obj.new_zeros(expected)
    t, o, d = (min(obj.shape[i], expected[i]) for i in range(3))
    fixed[:t, :o, :d] = obj[:t, :o, :d]
    items[1] = torch.nan_to_num(fixed, nan=0.0, posinf=0.0, neginf=0.0)
    return tuple(items)

def _object_safe_collate(batch):
    return default_collate([_fit_object_feature_sample(sample) for sample in batch])

for _name, _dataset in [('train', train_ds), ('val', val_ds), ('test', test_ds)]:
    if len(_dataset) == 0:
        raise RuntimeError(f'{_name} dataset is empty; verify CELL 3 paths.')

_raw_probe = train_ds[0]
_fixed_probe = _fit_object_feature_sample(_raw_probe)
print('Object feature guard:', tuple(_raw_probe[1].shape), '->', tuple(_fixed_probe[1].shape))
assert tuple(_fixed_probe[1].shape) == (args.topK_frame, args.objs, args.obj_feat_dim)
del _raw_probe, _fixed_probe

train_loader = DataLoader(train_ds, args.bs, shuffle=True, num_workers=0,
                          pin_memory=False, collate_fn=_object_safe_collate)
val_loader = DataLoader(val_ds, args.bs, shuffle=False, num_workers=0,
                        pin_memory=False, collate_fn=_object_safe_collate)
test_loader = DataLoader(test_ds, args.bs, shuffle=False, num_workers=0,
                         pin_memory=False, collate_fn=_object_safe_collate)
print('Colab DataLoaders: bs=32, num_workers=0, pin_memory=False, object_dim=2820')

train_sample_keys = [f"{row.video_id}_{row.type}" for row in train_ds.sample_list.itertuples(index=False)]
train_key_to_idx = {k: i for i, k in enumerate(train_sample_keys)}

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

In [ ]:
# CELL 7: Model + Optimizers + NCOD U + Generic Improvements
print('=== CELL 7: Model ===')
cfg = {k: v for k, v in Config.__dict__.items() if not k.startswith('_')}
cfg['device'] = device
cfg['topK_frame'] = args.select_frames
cfg['lambda_knowledge'] = 0.0  # keep the repository knowledge path disabled
model = VideoQAmodel(**cfg)
# The fixed run1 never calls the repository knowledge path. Remove these modules so
# they consume no optimizer/EMA/checkpoint space; HUM family IDs stay outside model.
for module_name in ('q_family_embed', 'knowledge_head', 'k_proj'):
    if hasattr(model, module_name):
        delattr(model, module_name)
model.to(device)

ema_model = None
if args.use_ema:
    ema_model = copy.deepcopy(model).to(device)
    ema_model.eval()
    for p in ema_model.parameters():
        p.requires_grad_(False)

non_text_params = []
text_base_params = []
for name, p in model.named_parameters():
    if not p.requires_grad:
        continue
    if 'text_encoder' in name:
        text_base_params.append(p)
    else:
        non_text_params.append(p)

param_groups = []
if len(non_text_params) > 0:
    param_groups.append({'params': non_text_params, 'lr': args.lr, 'weight_decay': args.decay})
if len(text_base_params) > 0:
    param_groups.append({'params': text_base_params, 'lr': args.text_encoder_lr, 'weight_decay': args.decay})
if len(param_groups) == 0:
    raise RuntimeError('No trainable parameters found for optimizer_model.')

optimizer_model = torch.optim.AdamW(param_groups)

steps_per_epoch = max(1, math.ceil(len(train_loader) / args.accumulation_steps))
total_steps = max(1, args.epoch * steps_per_epoch)
warmup_steps = max(1, args.warmup_epochs * steps_per_epoch)

def _lr_lambda(step):
    if step < warmup_steps:
        return float(step + 1) / float(warmup_steps)
    progress = float(step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    cosine = 0.5 * (1.0 + math.cos(math.pi * min(1.0, progress)))
    min_ratio = args.min_lr / max(args.lr, 1e-12)
    return max(min_ratio, cosine)

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer_model, lr_lambda=_lr_lambda)

U = torch.nn.Parameter(
    torch.full((len(train_ds),), 1e-8, dtype=torch.float32, device=device)
)
optimizer_u = torch.optim.SGD([U], lr=args.ncod_u_lr)

bce = nn.BCEWithLogitsLoss()

save_path = os.path.join(MODEL_DIR, MODEL_FILENAME)

print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.2f}M')
print(f'Text-encoder trainable params: {sum(p.numel() for p in text_base_params)/1e6:.3f}M')
print(f'EMA enabled: {ema_model is not None}')
print(f'Scheduler: {args.lr_schedule}')
print(f'U shape: {tuple(U.shape)}')
print(f'Artifacts: best={BEST_ARTIFACT_NAME} | latest={LAST_ARTIFACT_NAME}')

In [ ]:
# CELL 8: Init W&B + Resume Checkpoint
print('=== CELL 8: Initialize W&B Run ===')

start_epoch = 1
best_acc = 0.0
best_epoch = 0
epochs_without_improvement = 0
history = {'train_loss': [], 'train_acc': [], 'val_acc': []}
history_rows = []

LATEST_CKPT_PATH = os.path.join(MODEL_DIR, LATEST_CKPT_FILENAME)
TRAIN_HISTORY_CSV_PATH = os.path.join(MODEL_DIR, TRAIN_HISTORY_FILENAME)

RESUME_FROM_CHECKPOINT = True
RESUME_SOURCE = 'wandb' if WANDB_MODE_SETTING == 'online' else 'local'
RESUME_ARTIFACT_ALIAS = 'latest'
LOCAL_RESUME_PATH = LATEST_CKPT_PATH

wandb_config = {
    'run_tag': RUN_TAG,
    'run1_fixed_config': True,
    'backbone': 'dinov3+groundingdino',
    'text_encoder': args.text_encoder_type,
    'full_text_finetune': not args.freeze_text_encoder,
    'physical_batch_size': args.bs,
    'accumulation_steps': args.accumulation_steps,
    'effective_batch_size': args.bs * args.accumulation_steps,
    'epochs': args.epoch,
    'lambda_verifier': args.lambda_verifier,
    'ncod_u_lr': args.ncod_u_lr,
    'hum_alpha': args.hum_alpha,
    'lr_main': args.lr,
    'lr_text_encoder': args.text_encoder_lr,
    'lr_schedule': args.lr_schedule,
    'warmup_epochs': args.warmup_epochs,
    'min_lr': args.min_lr,
    'use_hard_neg_mining': args.use_hard_neg_mining,
    'hard_neg_weight_max': args.hard_neg_weight_max,
    'use_ema': args.use_ema,
    'ema_decay': args.ema_decay,
    'validation_metric': 'Acc_ALL_like_test',
    'early_stop_patience': args.early_stop_patience,
    'early_stop_min_delta': args.early_stop_min_delta,
    'early_stop_start_epoch': args.early_stop_start_epoch,
    'resume_enabled': RESUME_FROM_CHECKPOINT,
    'resume_source': RESUME_SOURCE,
    'platform': 'colab'
}

run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name=RUN_TAG, config=wandb_config, reinit=True)
wandb.watch(model, log='gradients', log_freq=100)
print(f'W&B run: {run.url if run else "(offline)"}')

def _load_resume_checkpoint(path, map_location):
    if not os.path.exists(path):
        raise FileNotFoundError(f'Checkpoint not found: {path}')
    return torch.load(path, map_location=map_location)

if RESUME_FROM_CHECKPOINT:
    print(f'Resume enabled from: {RESUME_SOURCE}')
    try:
        checkpoint = None
        resume_path = None

        if RESUME_SOURCE == 'local':
            resume_path = LOCAL_RESUME_PATH
            checkpoint = _load_resume_checkpoint(resume_path, device)
        elif RESUME_SOURCE == 'wandb':
            api = wandb.Api()
            resume_entity = WANDB_ENTITY or api.default_entity
            artifact_path = f'{resume_entity}/{WANDB_PROJECT}/{LAST_ARTIFACT_NAME}:{RESUME_ARTIFACT_ALIAS}'
            print(f'Downloading artifact: {artifact_path}')
            artifact = api.artifact(artifact_path)
            artifact_dir = artifact.download()
            candidate_path = os.path.join(artifact_dir, LATEST_CKPT_FILENAME)
            if os.path.exists(candidate_path):
                resume_path = candidate_path
            else:
                ckpt_files = [f for f in os.listdir(artifact_dir) if f.endswith('.ckpt')]
                if not ckpt_files:
                    raise FileNotFoundError(f'No .ckpt found in artifact folder: {artifact_dir}')
                resume_path = os.path.join(artifact_dir, ckpt_files[0])
            checkpoint = _load_resume_checkpoint(resume_path, device)
        else:
            raise ValueError("RESUME_SOURCE must be 'local' or 'wandb'")

        ckpt_state = checkpoint['model_state_dict']
        model_state = model.state_dict()
        filtered_state = {}
        skipped_keys = []
        for k, v in ckpt_state.items():
            if k in model_state and v.shape != model_state[k].shape:
                skipped_keys.append(f'{k}: ckpt={list(v.shape)} vs model={list(model_state[k].shape)}')
            else:
                filtered_state[k] = v

        if skipped_keys:
            print(f'Skipped {len(skipped_keys)} keys due to shape mismatch')

        missing, unexpected = model.load_state_dict(filtered_state, strict=False)
        if missing:
            print(f'Warning: missing model keys when resume: {len(missing)}')
        if unexpected:
            print(f'Warning: unexpected model keys when resume: {len(unexpected)}')

        if not skipped_keys:
            if 'optimizer_model_state_dict' in checkpoint:
                optimizer_model.load_state_dict(checkpoint['optimizer_model_state_dict'])
            if 'optimizer_u_state_dict' in checkpoint:
                optimizer_u.load_state_dict(checkpoint['optimizer_u_state_dict'])
            if 'scheduler_state_dict' in checkpoint:
                scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
            if ema_model is not None and checkpoint.get('ema_model_state_dict') is not None:
                ema_model.load_state_dict(checkpoint['ema_model_state_dict'], strict=False)
            if 'U' in checkpoint:
                with torch.no_grad():
                    u_ckpt = checkpoint['U'].to(device).float().view(-1)
                    n = min(u_ckpt.numel(), U.numel())
                    U[:n].copy_(u_ckpt[:n])

            best_acc = float(checkpoint.get('best_acc', 0.0))
            start_epoch = int(checkpoint.get('epoch', 0)) + 1
            best_epoch = int(checkpoint.get('best_epoch', 0))
            epochs_without_improvement = int(checkpoint.get('epochs_without_improvement', 0))
            history = checkpoint.get('history', history)
            history_rows = checkpoint.get('history_rows', history_rows)

            if os.path.exists(TRAIN_HISTORY_CSV_PATH):
                try:
                    history_rows = pd.read_csv(TRAIN_HISTORY_CSV_PATH).to_dict('records')
                    print(f'Loaded history CSV with {len(history_rows)} rows')
                except Exception as csv_err:
                    print(f'Warning: failed to load history CSV: {csv_err}')
        else:
            print('Optimizer/scheduler/U NOT restored due to architecture change.')

        print(f'Resumed from: {resume_path}')
        print(f'Start epoch: {start_epoch} | Best acc: {best_acc:.2f}% | Best epoch: {best_epoch}')
    except Exception as e:
        print(f'Warning: resume failed, starting from scratch. Error: {e}')
else:
    print('Resume disabled. Training starts from epoch 1.')

In [ ]:
# CELL 9: Training + local last/best checkpoints + metric logging
print('=== CELL 9: Training ===')

if RUN_TRAINING:
    stop_training = False
    for ep in range(start_epoch, args.epoch + 1):
        print(f'\nEpoch {ep}/{args.epoch}')

        # Verifier warmup: keep the auxiliary verifier loss off for two epochs
        if ep <= args.aux_warmup_epochs:
            eff_lambda_verifier = 0.0
        else:
            eff_lambda_verifier = args.lambda_verifier

        total_loss, l1, l2, verifier_loss, train_acc = train_epoch_integrated(
            model=model,
            optimizer_model=optimizer_model,
            optimizer_u=optimizer_u,
            U=U,
            loader=train_loader,
            bce=bce,
            device=device,
            epoch=ep,
            key_to_idx=train_key_to_idx,
            accumulation_steps=args.accumulation_steps,
            hum_alpha=args.hum_alpha,
            lambda_verifier=eff_lambda_verifier,
            use_hard_neg_mining=args.use_hard_neg_mining,
            hard_neg_weight_max=args.hard_neg_weight_max,
            ema_model=ema_model,
            ema_decay=args.ema_decay,
            scheduler=scheduler,
            scheduler_step_per_batch=True
        )

        eval_model = ema_model if ema_model is not None else model
        val_metrics = eval_epoch(eval_model, val_loader, device)
        val_acc = float(val_metrics['Acc_ALL'])
        val_plain_acc = float(val_metrics['Plain_Acc'])
        history['train_loss'].append(total_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        current_lrs = [pg['lr'] for pg in optimizer_model.param_groups]
        min_lr_now = float(min(current_lrs))
        max_lr_now = float(max(current_lrs))

        improved = val_acc > (best_acc + args.early_stop_min_delta)
        if improved:
            best_acc = val_acc
            best_epoch = ep
            epochs_without_improvement = 0
            print(f'New best val_acc(Acc_ALL)={best_acc:.2f}% at epoch {best_epoch} | val_plain_acc={val_plain_acc:.2f}%')
        elif ep >= args.early_stop_start_epoch:
            epochs_without_improvement += 1
            print(
                f'No significant improvement for {epochs_without_improvement} epoch(s) '
                f'(patience={args.early_stop_patience}, min_delta={args.early_stop_min_delta})'
            )

        epoch_row = {
            'epoch': ep,
            'train_total_loss': float(total_loss),
            'train_l1': float(l1),
            'train_l2': float(l2),
            'train_verifier_loss': float(verifier_loss),
            'train_acc': float(train_acc),
            'val_acc': float(val_acc),
            'val_plain_acc': float(val_plain_acc),
            'val_Description': float(val_metrics.get('Description', 0.0)),
            'val_Explanation': float(val_metrics.get('Explanation', 0.0)),
            'val_PAR': float(val_metrics.get('PAR', 0.0)),
            'val_CAR': float(val_metrics.get('CAR', 0.0)),
            'lambda_verifier_eff': float(eff_lambda_verifier),
            'u_mean': float(U.detach().mean().item()),
            'u_max': float(U.detach().max().item()),
            'lr_main_min': min_lr_now,
            'lr_main_max': max_lr_now,
            'best_acc_so_far': float(best_acc),
            'best_epoch_so_far': int(best_epoch),
            'epochs_without_improvement': int(epochs_without_improvement)
        }
        history_rows.append(epoch_row)
        pd.DataFrame(history_rows).to_csv(TRAIN_HISTORY_CSV_PATH, index=False)

        wandb.log(epoch_row)

        ckpt = {
            'epoch': ep,
            'model_state_dict': model.state_dict(),
            'ema_model_state_dict': ema_model.state_dict() if ema_model is not None else None,
            'optimizer_model_state_dict': optimizer_model.state_dict(),
            'optimizer_u_state_dict': optimizer_u.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_acc': best_acc,
            'best_epoch': best_epoch,
            'epochs_without_improvement': epochs_without_improvement,
            'history': history,
            'history_rows': history_rows,
            'U': U.detach().cpu(),
            'train_sample_keys': train_sample_keys
        }

        torch.save(ckpt, LATEST_CKPT_PATH)

        if improved:
            torch.save(ckpt, save_path)
        if ep >= args.early_stop_start_epoch and epochs_without_improvement >= args.early_stop_patience:
            print(f'Early stopping at epoch {ep}. Best val_acc(Acc_ALL)={best_acc:.2f}% at epoch {best_epoch}.')
            if wandb.run is not None:
                wandb.run.summary['early_stopped'] = True
                wandb.run.summary['early_stop_epoch'] = int(ep)
            stop_training = True
            break

    if wandb.run is not None:
        wandb.run.summary['best_val_acc'] = float(best_acc)
        wandb.run.summary['best_epoch'] = int(best_epoch)

    if os.path.exists(save_path):
        best_ckpt = torch.load(save_path, map_location=device)
        best_state = best_ckpt.get('ema_model_state_dict') or best_ckpt['model_state_dict']
        model.load_state_dict(best_state, strict=False)
        if ema_model is not None and best_ckpt.get('ema_model_state_dict') is not None:
            ema_model.load_state_dict(best_ckpt['ema_model_state_dict'], strict=False)
        print(f'Loaded best checkpoint from epoch {best_epoch} for final evaluation.')

    if not stop_training:
        print(f'Training finished all {args.epoch} epochs. Best Val Acc_ALL: {best_acc:.2f}%')
else:
    print('Skipping training')

In [ ]:
# CELL 10: Detailed model-only evaluation and CSV export
print('=== CELL 10: Detailed Evaluation (no knowledge JSON, no memory gate) ===')

CSV_OUTPUT_PATH = os.path.join(MODEL_DIR, PREDICTIONS_CSV_FILENAME)
COMPARISON_CSV_PATH = os.path.join(MODEL_DIR, 'run1_comparison.csv')

def _build_eval_meta_map(loader):
    sample_list = getattr(getattr(loader, 'dataset', None), 'sample_list', None)
    meta_map = {}
    if sample_list is None:
        return meta_map
    for _, row in sample_list.iterrows():
        video_id = str(row.get('video_id', ''))
        qtype = str(row.get('type', 'unknown'))
        meta_map[f'{video_id}_{qtype}'] = {
            'video_id': video_id,
            'question_type': qtype,
            'question': str(row.get('question', '')),
            'answers': [str(row.get(f'a{i}', '')) for i in range(5)],
        }
    return meta_map

def evaluate_detailed_run1(model, loader, device, log_to_wandb=True):
    model.eval()
    type_results = {}
    prediction_rows = []
    meta_map = _build_eval_meta_map(loader)

    with torch.inference_mode():
        for batch in tqdm(loader, desc='Run1 test evaluation'):
            ff, of, qns, ans_word, ans_id, qns_keys, q_family_id = _unpack_batch(batch)
            ff, of = ff.to(device), of.to(device)
            out = model(ff, of, qns, ans_word, return_aux=True, q_family_id=None)
            probs = torch.softmax(out['logits'], dim=-1).cpu().numpy()
            predictions = probs.argmax(axis=-1)
            targets = ans_id.numpy()

            for key, pred, target, prob_vec in zip(qns_keys, predictions, targets, probs):
                meta = meta_map.get(str(key), {})
                qtype = str(meta.get('question_type', 'unknown'))
                video_id = str(meta.get('video_id', str(key)))
                answers = list(meta.get('answers', ['', '', '', '', '']))[:5]
                answers += [''] * (5 - len(answers))
                pred, target = int(pred), int(target)
                correct = bool(pred == target)

                type_results.setdefault(qtype, []).append({
                    'video_id': video_id, 'correct': correct,
                })
                row = {
                    'video_id': video_id,
                    'question_type': qtype,
                    'question': str(meta.get('question', '')),
                    'correct_idx': target,
                    'predicted_idx': pred,
                    'is_correct': int(correct),
                    'confidence': float(prob_vec[pred]),
                    'predicted_answer': answers[pred],
                    'correct_answer': answers[target],
                }
                for i in range(5):
                    row[f'a{i}'] = answers[i]
                    row[f'prob_a{i}'] = float(prob_vec[i])
                prediction_rows.append(row)

    if not prediction_rows:
        raise RuntimeError('Evaluation produced zero predictions.')
    prediction_df = pd.DataFrame(prediction_rows)
    prediction_df.to_csv(CSV_OUTPUT_PATH, index=False)

    metrics = _compute_acc_all_metrics(type_results)
    metrics['Plain_Acc'] = 100.0 * prediction_df['is_correct'].mean()
    metrics['WeightedScore_WeakPriority'] = (
        0.35 * metrics['Predictive-Reason']
        + 0.35 * metrics['Counterfactual-Reason']
        + 0.20 * metrics['Acc_ALL']
        + 0.10 * float(best_acc)
    )

    if log_to_wandb and wandb.run is not None:
        wandb.log({f'eval/{name.replace("-", "_")}': float(value) for name, value in metrics.items()})
    print(f"PAR={metrics['PAR']:.2f}% | CAR={metrics['CAR']:.2f}% | Acc_ALL={metrics['Acc_ALL']:.2f}%")
    print(f'Saved predictions: {CSV_OUTPUT_PATH}')
    return metrics, type_results, CSV_OUTPUT_PATH

metrics, raw_results, predictions_csv = evaluate_detailed_run1(
    model, test_loader, device, log_to_wandb=True
)

comparison_row = {
    'run_tag': RUN_TAG,
    'best_val_acc': float(best_acc),
    'best_epoch': int(best_epoch),
    **{name: float(value) for name, value in metrics.items()},
}
if os.path.exists(COMPARISON_CSV_PATH):
    comparison_df = pd.read_csv(COMPARISON_CSV_PATH)
    comparison_df = comparison_df[comparison_df['run_tag'] != RUN_TAG]
    comparison_df = pd.concat([comparison_df, pd.DataFrame([comparison_row])], ignore_index=True)
else:
    comparison_df = pd.DataFrame([comparison_row])
comparison_df.to_csv(COMPARISON_CSV_PATH, index=False)

if wandb.run is not None:
    wandb.run.summary.update({
        'run_tag': RUN_TAG,
        'weighted_score_weak_priority': float(metrics['WeightedScore_WeakPriority']),
    })
print(metrics)


In [ ]:
# CELL 11: Upload exactly one last weight and one best weight, then finish W&B
print('=== CELL 11: Finish W&B ===')

METRICS_JSON_PATH = os.path.join(MODEL_DIR, METRICS_JSON_FILENAME)
with open(METRICS_JSON_PATH, 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'Saved metrics JSON: {METRICS_JSON_PATH}')

UPLOAD_CKPT_ARTIFACTS_AT_FINISH = True
if UPLOAD_CKPT_ARTIFACTS_AT_FINISH and wandb.run is not None:
    if os.path.exists(LATEST_CKPT_PATH):
        latest_ckpt_artifact = wandb.Artifact(
            name=LAST_ARTIFACT_NAME,
            type='model',
            metadata={
                'stage': 'finish',
                'checkpoint_kind': 'last',
                'run_tag': RUN_TAG,
                'text_encoder': args.text_encoder_type,
                'ncod_hum': True
            }
        )
        latest_ckpt_artifact.add_file(LATEST_CKPT_PATH, name=LATEST_CKPT_FILENAME)
        wandb.log_artifact(latest_ckpt_artifact, aliases=['last', 'latest'])
        print('Uploaded last weight to W&B exactly once.')
    else:
        print(f'Warning: last checkpoint not found at {LATEST_CKPT_PATH}')

    if os.path.exists(save_path):
        best_ckpt_artifact = wandb.Artifact(
            name=BEST_ARTIFACT_NAME,
            type='model',
            metadata={
                'stage': 'finish',
                'checkpoint_kind': 'best',
                'run_tag': RUN_TAG,
                'text_encoder': args.text_encoder_type,
                'ncod_hum': True
            }
        )
        best_ckpt_artifact.add_file(save_path, name=MODEL_FILENAME)
        wandb.log_artifact(best_ckpt_artifact, aliases=['best'])
        print('Uploaded best weight to W&B exactly once.')
    else:
        print(f'Warning: best checkpoint not found at {save_path}')

if wandb.run is not None:
    final_artifact = wandb.Artifact(
        name=FINAL_ARTIFACT_NAME,
        type='results',
        metadata={
            'run_tag': RUN_TAG,
            'backbone': 'dinov3+groundingdino',
            'text_encoder': args.text_encoder_type,
            'ncod_hum': True,
            'platform': 'colab'
        }
    )
    if os.path.exists(METRICS_JSON_PATH):
        final_artifact.add_file(METRICS_JSON_PATH, name=METRICS_JSON_FILENAME)
    if os.path.exists(predictions_csv):
        final_artifact.add_file(predictions_csv, name=PREDICTIONS_CSV_FILENAME)
    if os.path.exists(TRAIN_HISTORY_CSV_PATH):
        final_artifact.add_file(TRAIN_HISTORY_CSV_PATH, name=TRAIN_HISTORY_FILENAME)

    comparison_csv_path = os.path.join(MODEL_DIR, 'run1_comparison.csv')
    if os.path.exists(comparison_csv_path):
        final_artifact.add_file(comparison_csv_path, name='run1_comparison.csv')

    wandb.log_artifact(final_artifact)
    wandb.finish()
print('W&B finished: metrics/results plus exactly one last and one best weight artifact.')